# 06 · Portfolio Transaction Simulation

Replays transactions through the FIFO lot engine, reconciles derived holdings
against a broker snapshot, and produces a per-security portfolio report.

**This notebook uses `src/portfolio_sim.py`.  
It does not import from `src/tax_risk_sim.py` or `src/inputs.py`.**

---

## Data sources

Set `MODE` in the next cell to choose the data source:

| Mode | Source | When to use |
|------|--------|-------------|
| `"pdf"` | Deutsche Bank PDF in `data/private/` | Real portfolio — parses transactions + holdings directly from the broker PDF |
| `"synthetic"` | Bundled CSV fixtures in `data/examples/` | Testing, CI, offline demos — no PDF or internet required |

**PDF mode** seeds the lot ledger from the holdings snapshot (covers positions
bought before the report period), then replays all transactions from the PDF on
top. Reconciliation confirms whether the lot engine's share counts match the
broker snapshot.

> **FX note**: PDF mode uses live ECB rates (internet required).
> Synthetic mode uses a fixed-rate stub (fully offline).

In [1]:
import os
import sys
from pathlib import Path

# ── Mode selector ─────────────────────────────────────────────────────────────
# "pdf"       → parse a Deutsche Bank PDF (real data, requires data/private/)
# "synthetic" → use bundled synthetic fixtures (no PDF needed, works offline)
MODE = "pdf"

WORK = Path(".").resolve()
if WORK.name == "notebooks":
    WORK = WORK.parent
SRC = WORK / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# ── PDF mode paths (read from env — never hardcode data/private/ filenames) ───
if MODE == "pdf":
    _pdf_env = os.environ.get("DB_PDF_PATH")
    if not _pdf_env:
        raise EnvironmentError(
            "Set DB_PDF_PATH to your Deutsche Bank PDF report path.\n"
            "Example:  export DB_PDF_PATH=data/private/report.pdf"
        )
    PDF_PATH = Path(_pdf_env) if Path(_pdf_env).is_absolute() else WORK / _pdf_env
    TICKER_MAP_PATH = WORK / "data/private/ticker_map.json"

# ── Synthetic mode paths ──────────────────────────────────────────────────────
TX_CSV      = WORK / "data/examples/db_transactions_synthetic.csv"
HLD_CSV     = WORK / "data/examples/db_holdings_synthetic.csv"
TX_PARQUET  = WORK / "data/examples/db_transactions_synthetic.parquet"
HLD_PARQUET = WORK / "data/examples/db_holdings_synthetic.parquet"

print(f"Mode      : {MODE}")
print(f"Repo root : {WORK}")
if MODE == "pdf":
    print(f"PDF       : {PDF_PATH}")

Mode      : pdf
Repo root : /home/jovyan/work
PDF       : /home/jovyan/work/data/private/broker-report.pdf


In [2]:
import json
import subprocess

import pandas as pd

if MODE == "pdf":
    # ── Parse Deutsche Bank PDF ───────────────────────────────────────────────
    from pdf_parser import parse_db_pdf

    _tx_df, _hld_df = parse_db_pdf(PDF_PATH)
    transactions      = _tx_df
    holdings_snapshot = _hld_df

    with open(TICKER_MAP_PATH) as _f:
        _raw = json.load(_f)
    TICKER_MAP = {k: v for k, v in _raw.items() if not k.startswith("_")}

    REPORTING_DATE = str(_hld_df["date"].iloc[0])

    print(f"Transactions  : {len(transactions)} rows")
    print(f"  range       : {transactions['date'].min()} → {transactions['date'].max()}")
    print(f"Holdings      : {len(holdings_snapshot)} positions")
    print(f"Reporting date: {REPORTING_DATE}")
    print(f"Ticker map    : {len(TICKER_MAP)} ISINs")

else:
    # ── Normalize synthetic CSVs → Parquet ───────────────────────────────────
    NORMALIZE = WORK / "scripts/normalize_portfolio_inputs.py"
    import sys as _sys

    for csv_path, pq_path, schema_type in [
        (TX_CSV,  TX_PARQUET,  "transactions"),
        (HLD_CSV, HLD_PARQUET, "holdings"),
    ]:
        if not pq_path.exists():
            result = subprocess.run(
                [
                    _sys.executable, str(NORMALIZE),
                    "--input",          str(csv_path),
                    "--output-csv",     str(csv_path.with_suffix(".normalized.csv")),
                    "--output-parquet", str(pq_path),
                    "--type",           schema_type,
                ],
                capture_output=True, text=True,
            )
            if result.returncode != 0:
                print(f"ERROR: {result.stderr}")
            else:
                print(result.stdout.strip())
        else:
            print(f"Already exists: {pq_path.name}")

    transactions      = pd.read_parquet(TX_PARQUET)
    holdings_snapshot = pd.read_parquet(HLD_PARQUET)
    REPORTING_DATE    = "2023-12-31"
    TICKER_MAP        = {}

    print(f"\nTransactions : {len(transactions)} rows")
    print(f"Holdings     : {len(holdings_snapshot)} rows")

Transactions  : 128 rows
  range       : 2024-01-02 → 2026-06-05
Holdings      : 14 positions
Reporting date: 2026-06-06
Ticker map    : 14 ISINs


In [3]:
print("Transactions:")
display(transactions)
print("\nHoldings snapshot:")
display(holdings_snapshot)

Transactions:


,date,isin,wkn,asset_name,transaction_type,quantity,price,currency,amount,fees,tax_withheld,jurisdiction
0,2024-01-02,US1912161007,850663,"COCA-COLACO.,THEREGISTERED SHARESDL-,25",buy,100.0,54.070,EUR,5428.35,0.0,0.0,US
1,2024-01-02,US5949181045,870747,"MICROSOFTCORP.RG.SH.DL-,00000625",buy,10.0,335.250,EUR,3369.02,0.0,0.0,US
2,2024-01-02,US67066G1040,918422,"NVIDIACORP.REGISTEREDSHARES DL-,001",buy,20.0,436.400,EUR,8759.56,0.0,0.0,US
3,2024-01-02,US92826C8394,A0NC7B,"VISAINC.REG.SHARESCLASSADL-,0001",buy,20.0,236.550,EUR,4750.45,0.0,0.0,US
4,2024-01-03,US4581401001,855681,"INTELCORP.REGISTEREDSHARESDL -,001",buy,100.0,43.755,EUR,4394.06,0.0,0.0,US
...,...,...,...,...,...,...,...,...,...,...,...,...
123,2026-06-01,US92826C8394,A0NC7B,"VISAINC.REG.SHARESCLASSADL-,0001",dividend,0.0,0.000,USD,6.81,0.0,0.0,US
124,2026-06-05,US0079031078,863186,"ADVANCEDMICRODEVICESINC.RG.SH. DL-,01",sell,25.0,415.200,EUR,8448.72,0.0,0.0,US
125,2026-06-05,IE00BTJRMP35,A12GVR,X(IE)-MSCIEM.MKTS1CDLFUNDS,buy,27.0,78.184,EUR,2117.87,0.0,0.0,IE
126,2026-06-05,IE00B945VV12,A1T8FS,VANG.FTSEDEV.EU.UETFEODFUNDS,buy,60.0,48.720,EUR,2930.10,0.0,0.0,IE



Holdings snapshot:


,date,isin,wkn,asset_name,quantity,price,currency,market_value,jurisdiction,cost_basis_eur
0,2026-06-06,NL0000226223,893438,"STMICROELECTRONICSN.V.AANDELENEO 1,04",20.0,61.13000,EUR,1222.60,NL,26.85450
1,2026-06-06,US0258161092,850226,"AMERICANEXPRESSCO.RG.SH.DL-,20",50.0,269.63735,EUR,13481.87,US,204.82260
2,2026-06-06,US1912161007,850663,"COCA-COLACO.,THEREGISTEREDSHARES DL-,25",100.0,68.98467,EUR,6898.47,US,54.28350
3,2026-06-06,US4581401001,855681,"INTELCORP.REGISTEREDSHARESDL-,001",100.0,85.91837,EUR,8591.84,US,43.94060
4,2026-06-06,US0378331005,865985,APPLEINC.REGISTEREDSHARESO.N.,85.0,266.94670,EUR,22690.47,US,140.23518
5,2026-06-06,US5949181045,870747,"MICROSOFTCORP.RG.SH.DL-,00000625",35.0,361.75288,EUR,12661.35,US,302.50229
6,2026-06-06,US0231351067,906866,"AMAZON.COMINC.REGISTEREDSHARES DL-,01",30.0,213.53306,EUR,6405.99,US,129.54500
7,2026-06-06,US67066G1040,918422,"NVIDIACORP.REGISTEREDSHARESDL-,001",205.0,178.04258,EUR,36498.73,US,63.38590
8,2026-06-06,DE000A0F5UF5,A0F5UF,ISHARE.NASDAQ-100UCITSETFDE INHABER-ANT.,340.0,247.70000,EUR,84218.00,DE,203.24971
9,2026-06-06,US92826C8394,A0NC7B,"VISAINC.REG.SHARESCLASSADL-,0001",20.0,280.84258,EUR,5616.85,US,237.52250


In [4]:
from portfolio_sim import ECBProvider, FXProvider, make_fx_provider

if MODE == "pdf":
    # Live ECB rates — requires internet
    fx = make_fx_provider("ecb")
    print("FX provider: ECBProvider (live rates)")
else:
    # Fixed-rate stub — offline, reproducible for synthetic demos
    class FixedRateFXProvider(FXProvider):
        _RATES: dict[tuple[str, str], float] = {
            ("USD", "EUR"): 0.9200,
            ("EUR", "USD"): 1.0870,
            ("GBP", "EUR"): 1.1500,
            ("EUR", "GBP"): 0.8696,
        }

        def rate(self, from_currency: str, to_currency: str, date: str) -> float:
            if from_currency == to_currency:
                return 1.0
            key = (from_currency, to_currency)
            if key in self._RATES:
                return self._RATES[key]
            to_eur  = self.rate(from_currency, "EUR", date)
            eur_to  = self.rate("EUR", to_currency, date)
            return to_eur * eur_to

    fx = FixedRateFXProvider()
    print("FX provider: FixedRateFXProvider (offline demo)")
    print(f"  USD → EUR : {fx.rate('USD', 'EUR', '2023-12-31'):.4f}")

FX provider: ECBProvider (live rates)


In [5]:
from portfolio_sim import YahooPriceProvider, fetch_current_prices

CAPITAL_GAINS_TAX = 0.26375   # German Abgeltungsteuer 25% + Solidaritätszuschlag
DIVIDEND_TAX      = 0.26375

if MODE == "pdf":
    # Fetch live EUR prices for every holding via Yahoo Finance
    _price_provider = YahooPriceProvider(isin_to_ticker=TICKER_MAP, fx_provider=fx)
    _isins = holdings_snapshot["isin"].dropna().unique().tolist()
    CURRENT_PRICES_EUR = fetch_current_prices(_isins, _price_provider, REPORTING_DATE)
    _missing = [i for i in _isins if i not in CURRENT_PRICES_EUR]
    print(f"Prices fetched : {len(CURRENT_PRICES_EUR)}/{len(_isins)} ISINs")
    if _missing:
        print(f"  WARNING: no price for {_missing} — those positions will be excluded")
else:
    # Static prices for the synthetic demo (end-2023 approximations)
    _apple_eur = fx.rate("USD", "EUR", REPORTING_DATE) * 192.50
    CURRENT_PRICES_EUR = {
        "DE00SYNTH001": 15.60,
        "IE00B4L5Y983": 87.20,
        "US00SYNTH003": _apple_eur,
    }

print(f"\nReporting date     : {REPORTING_DATE}")
print(f"Capital gains tax  : {CAPITAL_GAINS_TAX:.5f}")
print(f"Dividend tax       : {DIVIDEND_TAX:.5f}")
print("\nCurrent prices (EUR):")
for isin, price in CURRENT_PRICES_EUR.items():
    print(f"  {isin}: {price:.4f}")

Prices fetched : 14/14 ISINs

Reporting date     : 2026-06-06
Capital gains tax  : 0.26375
Dividend tax       : 0.26375

Current prices (EUR):
  NL0000226223: 60.7560
  US0258161092: 266.8900
  US1912161007: 68.2818
  US4581401001: 85.1976
  US0378331005: 264.0378
  US5949181045: 357.9639
  US0231351067: 211.3660
  US67066G1040: 176.2028
  DE000A0F5UF5: 249.8500
  US92826C8394: 277.9811
  US02079K3059: 316.6065
  US30303M1027: 509.4502
  LU0274209740: 99.9600
  IE00BTJRMP35: 79.1460


In [6]:
# ── Step 5: check for unsupported corporate actions ───────────────────────────

from portfolio_sim import check_unsupported_actions

blocked_isins = check_unsupported_actions(transactions)

if blocked_isins:
    print(f'WARNING: {len(blocked_isins)} ISIN(s) blocked by unsupported corporate actions:')
    for isin in blocked_isins:
        print(f'  - {isin}')
    print('Partial results will exclude these securities.')
else:
    print('No unsupported corporate actions — full simulation available.')

No unsupported corporate actions — full simulation available.


In [ ]:
if MODE == "pdf":
    # PDF transactions cover only the report period (e.g. 2024–2026). Pre-period
    # buys are missing, so replaying tx_df from scratch hits "not enough shares"
    # on sells. Instead, seed lots from the broker holdings snapshot (authoritative
    # end-state cost basis) and report the current position without replaying.
    from portfolio_sim import initialize_lots_from_holdings, simulate_from_snapshot
    import pandas as pd

    _initial_lots = initialize_lots_from_holdings(holdings_snapshot)
    result = simulate_from_snapshot(
        initial_lots=_initial_lots,
        new_transactions=pd.DataFrame(),
        current_prices_eur=CURRENT_PRICES_EUR,
        capital_gains_tax_rate=CAPITAL_GAINS_TAX,
        dividend_tax_rate=DIVIDEND_TAX,
        fx_provider=fx,
        reporting_date=REPORTING_DATE,
    )
    excluded = []
    _lots_for_ledger = _initial_lots.to_dict("records")
    print("PDF mode: portfolio state derived from broker holdings snapshot.")
    print(f"  {len(_initial_lots)} lots seeded from {len(holdings_snapshot)} holdings")
    print(f"  {len(transactions)} transactions shown as history (not replayed — pre-period buys may be absent)")
else:
    from portfolio_sim import simulate_portfolio_partial

    result, excluded = simulate_portfolio_partial(
        transactions=transactions,
        current_prices_eur=CURRENT_PRICES_EUR,
        capital_gains_tax_rate=CAPITAL_GAINS_TAX,
        dividend_tax_rate=DIVIDEND_TAX,
        fx_provider=fx,
        lot_method="fifo",
        reporting_date=REPORTING_DATE,
    )
    _lots_for_ledger = None  # built in step 8

if excluded:
    print(f"\nPARTIAL RESULTS — excluded ISINs (unsupported actions): {excluded}")
    print("Totals below do NOT represent the full portfolio.\n")

display(result)

In [ ]:
# ── Step 7: portfolio totals ──────────────────────────────────────────────────

totals = result[[
    'market_value_eur',
    'unrealised_gain_eur',
    'realised_gain_ytd_eur',
    'tax_paid_ytd_eur',
]].sum()

print(f'Portfolio report — {REPORTING_DATE}')
print('=' * 45)
print(f'  Market value (EUR)      : {totals["market_value_eur"]:>12,.2f}')
print(f'  Unrealised gain (EUR)   : {totals["unrealised_gain_eur"]:>12,.2f}')
print(f'  Realised gain YTD (EUR) : {totals["realised_gain_ytd_eur"]:>12,.2f}')
print(f'  Tax paid YTD (EUR)      : {totals["tax_paid_ytd_eur"]:>12,.2f}')

if excluded:
    print()
    print(f'  PARTIAL — {len(excluded)} ISIN(s) excluded: {excluded}')

In [ ]:
from portfolio_sim import (
    apply_buy,
    apply_sell_fifo,
    apply_split,
    derive_holdings_from_lots,
    reconcile_holdings,
)

if MODE == "pdf":
    # Lots were seeded directly from the holdings snapshot in step 6.
    # Reconciliation confirms the snapshot loaded correctly — it will always
    # match because the source is the same, but it validates schema and parsing.
    lots = _lots_for_ledger
    lot_holdings    = derive_holdings_from_lots(lots)
    broker_holdings = holdings_snapshot[["isin", "quantity"]].copy()
else:
    # Replay all transactions to reconstruct the lot ledger from scratch.
    lots = []
    for _, row in transactions.sort_values("date").iterrows():
        isin     = row["isin"]
        tx_type  = row["transaction_type"]
        currency = str(row["currency"])
        date     = str(row["date"])

        if tx_type == "buy":
            price_eur = fx.convert(float(row["price"]), currency, "EUR", date)
            lots = apply_buy(lots, isin, date, price_eur, float(row["quantity"]))
        elif tx_type == "sell":
            price_eur = fx.convert(float(row["price"]), currency, "EUR", date)
            lots, _, _ = apply_sell_fifo(lots, isin, float(row["quantity"]), price_eur, CAPITAL_GAINS_TAX)
        elif tx_type == "split":
            lots = apply_split(lots, isin, float(row["quantity"]))

    lot_holdings    = derive_holdings_from_lots(lots)
    broker_holdings = holdings_snapshot[["isin", "quantity"]].copy()

reconciliation = reconcile_holdings(lot_holdings, broker_holdings)
print("Holdings reconciliation:")
display(reconciliation)

mismatches = reconciliation[reconciliation["status"] == "mismatch"]
if mismatches.empty:
    print("All holdings match the broker snapshot (within tolerance).")
else:
    print(f"WARNING: {len(mismatches)} mismatch(es) — review before trusting simulation output.")
    display(mismatches)

In [ ]:
# ── Step 9: lot ledger with derived unrealised gain ───────────────────────────

from portfolio_sim import lots_to_dataframe

lot_df = lots_to_dataframe(lots)
lot_df['unrealised_gain_eur'] = lot_df.apply(
    lambda r: round(
        (CURRENT_PRICES_EUR.get(r['isin'], 0.0) - r['lot_price_eur']) * r['remaining_shares'], 2
    ),
    axis=1,
)

print('Open lot ledger with derived unrealised gain (computed at reporting date, not stored):')
display(lot_df)

## Known limitations (v1)

| Limitation | Note |
|------------|------|
| Flat tax rates | `CAPITAL_GAINS_TAX` and `DIVIDEND_TAX` applied uniformly. German Sparer-Pauschbetrag (€1,000/year exemption) is not modelled. |
| Solidarity surcharge | Not modelled separately — fold it into the flat rate if needed (26.375% = 25% × 1.055). |
| Reverse-split fractional shares | Retained at full precision; no cash distribution event generated. |
| Unsupported corporate actions | `merger`, `spin_off`, `option` block the affected ISIN from trusted totals. |
| Jurisdiction-aware tax | `jurisdiction` field captured but not yet used to vary tax logic by country. |
| Transaction-date FX | FX conversion uses the transaction date; intraday spot rates are not modelled. |

See `docs/portfolio-transaction-mvp/requirements-and-decisions.md` for full decisions and limitations.